# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Inspect available record sets in the dataset
record_sets = list(dataset.record_sets)
print('Record Sets (@id):')
for rs in record_sets:
    print(f"- {rs['@id']} : {rs.get('name','No name')}")

# Inspect fields within each record set
for rs in record_sets:
    print(f"\nFields for Record Set {rs['@id']}:")
    for field in rs.get('field', []):
        print(f"    {field['@id']} : {field.get('name','No name')}  (type: {field.get('dataType','Unknown')})")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set using @id
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Display columns for one of the main record sets (choose the first as example)
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Columns in record set '{main_rs_id}':")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
# Example EDA with GAD-7 scores field
# Set record set and numeric field by @id
record_set_id = main_rs_id

# Find available numeric fields
numeric_field_id = None
for rs in record_sets:
    if rs['@id'] == record_set_id:
        for field in rs.get('field', []):
            if field.get('dataType') in ['schema:Float', 'schema:Integer', 'schema:Number']:
                # Example: 'gad7_score' or similar
                numeric_field_id = field['@id']
                print(f"Using numeric field: {numeric_field_id}")
                break

# Apply threshold filtering (example threshold)
threshold = 8
if numeric_field_id and numeric_field_id in dataframes[record_set_id].columns:
    filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Choose a group field, e.g. 'schema:gender' or similar demographic field
    group_field_id = None
    for rs in record_sets:
        if rs['@id'] == record_set_id:
            for field in rs.get('field', []):
                if 'gender' in field.get('name','').lower():
                    group_field_id = field['@id']
                    break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, show distribution of a numeric mental health score or compare scores across demographic groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Distribution of numeric field
if numeric_field_id in dataframes[record_set_id].columns:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[record_set_id][numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field found, show comparison
    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=dataframes[record_set_id][group_field_id], y=dataframes[record_set_id][numeric_field_id])
        plt.title(f"{numeric_field_id} Scores by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to load, explore, and analyze the FAIR² Mental Health Survey dataset using `mlcroissant`. We reviewed the available record sets and fields by `@id`, extracted data into DataFrames, performed basic EDA, and visualized the distribution of a key mental health indicator. The dataset provides insights into demographic and psychological patterns within Kilifi County, Kenya, and can support public health research and interventions.